In [13]:
## MY VERSION
import cv2
import imutils

#detector
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

cap = cv2.VideoCapture('pedestrian.mp4')

while cap.isOpened():
    ret, image = cap.read()
    if ret: 
        image = imutils.resize(image, width = min(400, image.shape[1]))

        #detecting all the regions in the img that has a pedestrian inside it
        (regions, _) = hog.detectMultiScale(image,winStride=(4,4),padding=(4,4),scale=1.05)

        for (x,y,w,h) in regions:
            cv2.rectangle(image, (x,y), (x+w,y+h), (0,0,255),2)

            cv2.imshow("Image",image)
            if cv2.waitKey(40) & 0xFF == ord('q'):
                break
    else: 
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import imutils
import os

# --- THE ARCH LINUX FIX ---
# Why? Arch uses Wayland. Pip's OpenCV is built for X11.
# This forces the app to use 'xcb' (XWayland) so the window actually opens.
os.environ["QT_QPA_PLATFORM"] = "xcb"

# --- STEP 1: INITIALIZE THE BRAIN ---
# HOG (Histogram of Oriented Gradients) looks at the direction of light/shadows.
# Humans have a specific "gradient signature" (head, shoulders, legs).
hog = cv2.HOGDescriptor()

# Load the pre-trained 'People Detector'. 
# This is a Support Vector Machine (SVM) model that already knows what humans look like.
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

# --- STEP 2: SOURCE THE FEED ---
# Remember: I renamed 'pedestrian.mp4' to 'pedestrain.mp4' because of that typo!
video_path = 'pedestrain.mp4' 

if not os.path.exists(video_path):
    print(f"❌ Error: Check the filename in the folder!")
    exit()

# VideoCapture creates a pointer to the file. It doesn't load the whole thing at once.
cap = cv2.VideoCapture(video_path)

# UI Setup: Create a named window so we can resize it later.
window_name = "HOG Video Detection"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 800, 600)

# --- STEP 3: THE MAIN LOOP ---
# Videos are just a series of images (frames). We process them one by one.
while cap.isOpened():
    # .read() returns: 
    #   1. success (True/False)
    #   2. frame (The actual image matrix)
    success, frame = cap.read()
    
    if not success:
        break # End of video or corrupted frame

    # --- OPTIMIZATION ---
    # HOG is slow. Processing a 1080p frame frame will lag like crazy.
    # We shrink it to 400px width. Smaller image = fewer pixels to scan = faster FPS.
    frame = imutils.resize(frame, width=400)

    # --- THE MAGIC BIT: DETECTION ---
    # detectMultiScale scans the image at different sizes (scales).
    # scale=1.05: Scans at 105% increments. Lower (1.01) is more accurate but SLOW.
    # winStride: How many pixels the scanning window 'jumps'. (4,4) is standard.
    # padding: Adds a buffer around the scan window to catch people near edges.
    (regions, _) = hog.detectMultiScale(frame, 
                                        winStride=(4, 4),
                                        padding=(8, 8),
                                        scale=1.05)

    # --- VISUALIZATION ---
    # 'regions' is a list of [x, y, width, height] for every detected person.
    for (x, y, w, h) in regions:
        # Drawing a green box (0, 255, 0) with a thickness of 2.
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Update the window with the new frame
    cv2.imshow(window_name, frame)

    # --- THE HEARTBEAT ---
    # cv2.waitKey(1) is vital. It tells the GUI to refresh for 1ms.
    # If I used waitKey(0), it would freeze on every single frame!
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break # Press 'q' to kill the script

# --- CLEANUP ---
# Close the file pointer and kill the windows to free up RAM.
cap.release()
cv2.destroyAllWindows()
print("Process Complete.")